# 4. Cluster Data

*Goal: Group similar command lines together using unsupervised learning.*

In this notebook, we move from data engineering into true Machine Learning. We will take our dense embedding vectors and use two different clustering algorithms (DBSCAN and KMeans) to automatically group similar commands together without needing any pre-labeled data.

## Step 1 — Import Libraries

Import the required modules:
- `pandas` for DataFrame operations (I like to import as `pandas` as `pd` for short hand)
- `numpy` for working with arrays of embedding vectors (I like to import as `numpy` as `np` for short hand)
- `DBSCAN` and `KMeans` from `sklearn.cluster` — the two clustering algorithms used in this notebook
- `silhouette_score` from `sklearn.metrics` to evaluate the quality of KMeans cluster assignments

In [ ]:
import pandas as pd  # <--- DataFrame usage for dataset operations
import numpy as np  # <--- Numpy arrays for clustering algorithms
from sklearn.cluster import DBSCAN, KMeans  # <--- Our clustering algorithms
from sklearn.metrics import silhouette_score  # <--- To evaluate the quality of our KMeans clusters

## Step 2 — Load the Dataset

We need to load the dataset that now contains our generated embeddings.

Read the Parquet file `"_03_evtx_commands_with_embeddings.parquet"` into a variable called `dataframe`. If we remember from the previous notebook, this contains a `CommandLine` column and an `Embedding` column holding the dense vector for each command

In [ ]:
dataframe = pd.read_parquet("_03_evtx_commands_with_embeddings.parquet")

## Step 3 — Extract Unique Commands and Vectors

Machine Learning algorithms like DBSCAN and KMeans don't understand Pandas DataFrames directly; they require pure 2D mathematical matrices (NumPy arrays). Furthermore, we only want to cluster *unique* commands. If we cluster 5,000 identical `whoami` commands, it wastes computation and artificially skews the cluster density.


Creaet a variable called `unique_df` that is a DataFrame that simplifies the dataframe down to unique commands (the `CommandLine` column) by calling `drop_duplicates` and reseting the index.

Example:
```python
dataframe.drop_duplicates(subset="COLUMN TO DEDUPLICATE BY")\
    .reset_index(drop=True)
```

Related Docs:
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.drop_duplicates.html
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.reset_index.html

In [ ]:
unique_df = dataframe.drop_duplicates(subset="CommandLine")\
    .reset_index(drop=True)

Now that we have records that are unique by `CommandLine`, we need to create an array of unique command lines. Grab the `CommandLine` column / series and call `.to_numpy()` on it to collect said array into a variable called `unique_command_lines`.

Example:
```python
unique_df["<COLUMN>"].to_numpy()
```

In [ ]:
unique_command_lines = unique_df["CommandLine"].to_numpy()

Now lets create an array of unique command line vectors that line up one for one with the unique command lines. We will use the `numpy.stack()` function for this and assign the results to the `unique_command_lines_vectors` variable.

Example:
```python
np.stack(unique_df["<COLUMN THAT CONTAINS EMBEDDINGS>"])
```

Related Docs:
 - https://numpy.org/doc/stable/reference/generated/numpy.stack.html

In [ ]:
unique_command_lines_vectors = np.stack(unique_df["Embedding"])

> Deriving both arrays from the **same rows of the same DataFrame** guarantees they stay in sync — `unique_command_lines[i]` is always the command for `unique_command_lines_vectors[i]`, which in turn matches `labels_[i]` after fitting a clustering model.

## Step 4 — Cluster with DBSCAN

Fit a `DBSCAN` model on the unique embedding vectors. We use the following parameters but the DBSCAN algorithm has more parameters that can be used to experiment with result sets.

Note that different embedding models are going to effect clustering differently and its worth playing around with the clustering parameters.

> DBSCAN does **not** require you to specify the number of clusters in advance. Points that don't fit any cluster are labelled `-1` (noise). The printed count includes that noise label.

First, set two DBSCAN tuning variables that control how strict clustering is:

- `eps_dbscan = 0.56`
  - The neighborhood radius (distance threshold).
  - Two points are considered neighbors only if they are within this distance in embedding space.
  - Lower values make clustering stricter (more points may become noise), while higher values make clustering looser.

- `min_samples = 2`
  - The minimum number of nearby points required to form a dense region (a cluster core).
  - With `2`, very small clusters can still be formed.

These variables are used when initializing `DBSCAN(...)` in the next code cell to determine cluster formation and which points are labeled as noise (`-1`).

In [ ]:
eps_dbscan = 0.46
min_samples = 2

Create a `DBSCAN` model instance and store it in a variable named `dbscan`. Pass it the two parameters we defined above.

- Call the `DBSCAN` constructor from `sklearn.cluster`.
- Use the tuning variables you set in the previous cell (`min_samples` and `eps_dbscan`).

Example:
```python
DBSCAN(min_samples=min_samples, eps=eps_dbscan)
```

Related Docs:
 - https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html

In [ ]:
dbscan = DBSCAN(min_samples=min_samples, eps=eps_dbscan)

Now call `dbscan`'s `fit` function and pass the function the `unique_command_lines_vectors`.

Related Docs:
 - https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html#sklearn.cluster.DBSCAN.fit

In [ ]:
dbscan.fit(unique_command_lines_vectors)

Count the number of unclustered commands. You can do this with:

```python
list(dbscan.labels_).count(-1)
```

Conversly, get the number of clusters found with:

```python
len(set(dbscan.labels_))
```

In [ ]:
num_of_unclustered_commands = list(dbscan.labels_).count(-1)
number_of_clusters = len(set(dbscan.labels_))
print(f"Number of Clusters: {number_of_clusters}. Unclustered: {num_of_unclustered_commands}")

**Exercise 4.1: In this exercise, we will explore DBSCAN and how changing parameters affects the output.**
  
   **Form a Hypothesis:**

- What do you predict will happen to the number of clusters and the number of noise points (label -1) if you significantly increase the `eps` value (e.g., from 0.56 to 1.0)?
- What do you predict will happen if you increase the `min_samples` value (e.g., from 2 to 5) while keeping `eps` the same?,
  
**Run the Experiment:**
- First, run the cell above with the original parameters from the instructions to see the baseline result.
- Then, modify the `eps` and `min_samples` values one at a time, as described above. Rerun the cell for each change and observe the new \"Number of Clusters\". What do you observe?

## Step 5 — Inspect DBSCAN Cluster Sizes

After running a clustering algorithm, we need to understand its output distribution. If 99% of our data is lumped into a single cluster, our parameters are too loose. If we have thousands of tiny clusters, they are too strict. Viewing the sizes of the clusters helps us evaluate the shape and quality of the model we just ran.

> Remember, cluster `-1` contains commands that DBSCAN could not assign to any group.

Create a DataFrame called `dbscan_clustered_commands_df` that contains our unique commands and their associated clusters. You can grab the list of clusters from `dbscan.labels_` which line up with the `unique_command_lines` list.

Example:
```python
pd.DataFrame({
    "Command": <UNIQUE COMMAND LINES>, 
    "Cluster": <CLUSTER>
})
```

In [ ]:
# Create a DataFrame that contains our unique commands and their associated clusters
dbscan_clustered_commands_df = pd.DataFrame({
    "Command": unique_command_lines, 
    "Cluster": dbscan.labels_
})

If you want to see how many commands belong to each cluster, you could group the DataFrame `dbscan_clustered_commands_df` by `Cluster` and perform a count. Then you could transpose the aggregation showing you an friendly view.

Example:
```python
# View cluster counts out of curiosity
dbscan_clustered_commands_df.groupby(["Cluster"])\
    .agg("count")\
    .reset_index()\
    .rename(columns={"Command": "Unique Commands in Cluster"})\
    .transpose()
```

Related Docs:
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.agg.html
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.reset_index.html
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.transpose.html

In [ ]:
# View cluster counts out of curiosity
dbscan_clustered_commands_df.groupby(["Cluster"])\
    .agg("count")\
    .reset_index()\
    .rename(columns={"Command": "Unique Commands in Cluster"})\
    .transpose()

We can further experiment with refining unclustered commands by iterating over the commands in the -1 cluster and reclustering them with more relaxed parameters. Create this `refine_clusters_with_dbscan` function we could use to further cluster the unclustered commands.

In [ ]:
def refine_clusters_with_dbscan(_dbscan, _unique_command_lines_vectors, _eps_dbscan) -> np.ndarray:
    # Start with initial labels
    _max_eps = .81  # Maximum eps to try for reclustering
    _increment = 0.05  # Increment eps by 0.05 in each iteration
    _refined_labels = _dbscan.labels_.copy()

    # Track next available cluster id (ignore -1 noise)
    _existing_cluster_ids = [c for c in set(_refined_labels) if c != -1]
    _next_cluster_id = (max(_existing_cluster_ids) + 1) if _existing_cluster_ids else 0

    # Increment eps by 0.05 until 0.80, reclustering only unclustered commands
    for eps in np.arange(_eps_dbscan, _max_eps, _increment):
        _unclustered_idx = np.where(_refined_labels == -1)[0]
        if len(_unclustered_idx) == 0:
            break

        _sub_vectors = _unique_command_lines_vectors[_unclustered_idx]
        _sub_dbscan = DBSCAN(n_jobs=-1, min_samples=2, eps=float(round(eps, 2)))
        _sub_dbscan.fit(_sub_vectors)

        _sub_labels = _sub_dbscan.labels_
        for lbl in sorted(set(_sub_labels)):
            if lbl == -1:
                continue
            _member_idx = _unclustered_idx[_sub_labels == lbl]
            _refined_labels[_member_idx] = _next_cluster_id
            _next_cluster_id += 1

        _still_unclustered = int(np.sum(_refined_labels == -1))
        print(f"eps={eps:.2f} -> remaining unclustered: {_still_unclustered}")

    _num_of_unclustered_commands = int(np.sum(_refined_labels == -1))
    _number_of_clusters = len(set(_refined_labels)) - (1 if -1 in _refined_labels else 0)
    print(f"Final clusters: {_number_of_clusters}. Unclustered: {_num_of_unclustered_commands}")

    return _refined_labels

Create a variable named `refined_labels` that stores the new list of labes (clusters) by calling `refine_clusters_with_dbscan` and passing `dbscan`, `unique_command_lines_vectors`, and `eps_dbscan` variables.

In [ ]:
refined_labels = refine_clusters_with_dbscan(dbscan, unique_command_lines_vectors, eps_dbscan)

Now create a variable called `dbscan_clustered_commands_df_refined` which is a DataFrame that contains our unique commands and the refined clusters.

Example:
```python
pd.DataFrame({
    "Command": <UNIQUE COMMAND LINES>, 
    "Cluster": <CLUSTER>
})
```

In [ ]:
# Create a DataFrame that contains our unique commands and their associated clusters
dbscan_clustered_commands_df_refined = pd.DataFrame({
    "Command": unique_command_lines, 
    "Cluster": refined_labels
})

Lets do our aggregated view again to see how many we clusters we have now.

Example:
```python
# View cluster counts out of curiosity
dbscan_clustered_commands_df_refined.groupby(["Cluster"])\
    .agg("count")\
    .reset_index()\
    .rename(columns={"Command": "Unique Commands in Cluster"})\
    .transpose()
```

Related Docs:
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.agg.html
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.reset_index.html
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.transpose.html

In [ ]:
# View cluster counts out of curiosity
dbscan_clustered_commands_df_refined.groupby(["Cluster"])\
    .agg("count")\
    .reset_index()\
    .rename(columns={"Command": "Unique Commands in Cluster"})\
    .transpose()

**Exercise: Experiment**

Just like you experimented earlier with tweaking DBSCAN parameters, you can do the same thing in the `refine_clusters_with_dbscan` function by twaeking the `_max_eps` and `_increment` variables.

Export the data to a csv file if it makes exploring the sorted data easier for you.

## Step 6 — Explore a Specific DBSCAN Cluster

Unsupervised learning algorithms don't actually "read" English; they simply group numbers that are mathematically close together. As data scientists and security analysts, we must perform a qualitative "spot-check" to ensure that these mathematical groupings actually make semantic sense in the real world.

Set `cluster_number` to cluster you want to view. We will view the commands in this cluster.

In [ ]:
cluster_number = 78

Be curious about what your data looks like. Create a filtered DataFrame to view the results of the specified cluster number just to see what commands are in it. You don't need to assign the output to a variable, we just want Jupyter to show you what the result looks like.

Example:
```python
dbscan_clustered_commands_df[dbscan_clustered_commands_df["Cluster"] == cluster_number].value_counts()
```

In [ ]:
dbscan_clustered_commands_df[dbscan_clustered_commands_df["Cluster"] == cluster_number].value_counts()

## Step 7 — Map DBSCAN Labels Back to the Full DataFrame

Create a lookup dictionary named `cluster_map_dbscan` by zipping `unique_command_lines` with `refined_labels`, then wrapping the result with `dict(...)`.

This produces:
- Keys: each unique command string from `unique_command_lines`
- Values: the corresponding DBSCAN cluster label from `refined_labels`

You will use this dictionary in the next cell with `.map()` to assign each original row in `dataframe` its `Cluster_DBSCAN` value.

Example:
```python
dict(zip(<KEYS>, <VALUES>))
```

In [ ]:
cluster_map_dbscan = dict(zip(unique_command_lines, refined_labels))

Create a new column called `Cluster_DBSCAN` in `dataframe` by calling `.map()` on the `CommandLine` column with `cluster_map_dbscan` as the lookup.

This assigns a DBSCAN cluster label to **every row** in the original DataFrame — including duplicate command lines — by looking up each command in the dictionary we built in the previous cell.

Example:
```python
dataframe[<NEW MAPPED COLUMN>] = dataframe[<COLUMN TO MAP>].map(<MAPPING>)
```

In [ ]:
dataframe["Cluster_DBSCAN"] = dataframe["CommandLine"].map(cluster_map_dbscan)

## Step 8 — Cluster with the KMeans Algorithm

### What is KMeans?

Think of KMeans like sorting a pile of mixed LEGO bricks into buckets. You decide upfront how many buckets you want (say, 10), then the algorithm repeatedly:
1. Places a "center point" in each bucket.
2. Assigns every brick to whichever center it is closest to.
3. Moves each center to the middle of its assigned bricks.
4. Repeats until the assignments stop changing.

The result is `k` tidy groups where every item belongs to exactly one group.

### How is it different from DBSCAN?

| | DBSCAN | KMeans |
|-|-|-|
| **Number of clusters** | Figured out automatically | You must choose upfront (`k`) |
| **Noise / outliers** | Points that don't fit are labeled `-1` (noise) | Every point is forced into a cluster |
| **Cluster shape** | Can find irregular, oddly shaped clusters | Assumes clusters are roughly round / equal-sized |
| **Best for** | Exploratory analysis where you don't know how many groups exist | When you have a target number of groups and want clean, balanced assignments |

Because this requires us to specify the number of clusters ahead of time, lets create a function that can help us find a good number to use.

Creat this `find_optimal_k_for_kmeans` function that will:

Iterate over candidate cluster counts (20 – 300 in steps of 5) and fit a `KMeans` model for each.  
For every fit, compute the **silhouette score** — a measure of how well-separated the clusters are, ranging from -1 (poor) to +1 (perfect).

The `k` with the highest score is selected as the optimal number of clusters.

> Rembember, this is helpful because KMeans requires you to specify `k` upfront. The silhouette search automates what would otherwise be a manual trial-and-error process.

In [ ]:
def find_optimal_k_for_kmeans(vectors: np.ndarray) -> int:
    _best_k = None
    _best_score = -1

    _k_range = range(20, 300, 5)

    for k in _k_range:
        _kmeans = KMeans(n_clusters=k, random_state=42)
        _labels = _kmeans.fit_predict(vectors)
        _score = silhouette_score(vectors, _labels)

        print(f"K={k}, Silhouette Score={_score:.4f}")

        if _score > _best_score:
            _best_score = _score
            _best_k = k

    print(f"Optimal K: {_best_k} with Silhouette Score: {_best_score:.4f}")
    return _best_k

### Cluster with KMeans
Now that we have a method defined for calculating an optimal number of clusters. Lets run it and set a variable called `k`. Pass the `unique_command_lines_vectors` variable to the function.

In [ ]:
k = find_optimal_k_for_kmeans(unique_command_lines_vectors)

Create a `KMeans` model instance and store it in a variable named `kmeans`. Pass it the parameters we defined above.

- Call the `KMeans` constructor from `sklearn.cluster`.
- Use the number of clusters variable you set in the previous cell (`k`).

Example:
```python
KMeans(n_clusters=k)
```

Related Docs:
 - https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#kmeans

In [ ]:
kmeans = KMeans(n_clusters=k)

Now call `kmeans`'s `fit` function and pass the function the `unique_command_lines_vectors`.

Related Docs:
 - https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans.fit

In [ ]:
kmeans.fit(unique_command_lines_vectors)

Now create a variable called `clustered_commands_kmeans` which is a DataFrame that contains our unique commands (`unique_command_lines`) and the clusters (`kmeans.labels_`).

Example:
```python
pd.DataFrame({
    "Command": <UNIQUE COMMAND LINES>, 
    "Cluster": <CLUSTERS>
})
```

In [ ]:
# Create a DataFrame that contains our commands and their associated clusters
clustered_commands_kmeans = pd.DataFrame({
    "Command": unique_command_lines, 
    "Cluster": kmeans.labels_
})

Lets do our aggregated view again to see cluster counts.

Example:
```python
# View cluster counts out of curiosity
clustered_commands_kmeans.groupby(["Cluster"])\
    .agg("count")\
    .reset_index()\
    .rename(columns={"Command": "Unique Commands in Cluster"})\
    .transpose()
```

Related Docs:
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.agg.html
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.reset_index.html
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.transpose.html

In [ ]:
# View cluster counts
clustered_commands_kmeans.groupby(["Cluster"])\
    .agg("count")\
    .reset_index()\
    .rename(columns={"Command": "Unique Commands in Cluster"})\
    .transpose()

## Step 9 — Explore a Specific KMeans Cluster

Lets do the same thing we did for the DBSCAN Clusters and view the commans in a given cluster.

Set `cluster_number` to cluster you want to view. We will view the commands in this cluster.

Example:
```python
cluster_number = 18
```

In [ ]:
cluster_number = 18

Be curious about what your data looks like. Create a filtered DataFrame to view the results of the specified cluster number just to see what commands are in it. You don't need to assign the output to a variable, we just want Jupyter to show you what the result looks like.

Example:
```python
clustered_commands_kmeans[clustered_commands_kmeans["Cluster"] == cluster_number].value_counts()
```

In [ ]:
clustered_commands_kmeans[clustered_commands_kmeans["Cluster"] == cluster_number].value_counts()

## Step 10 — Map KMeans Labels Back to the Full DataFrame

Create a lookup dictionary named `cluster_map_kmeans` by zipping `unique_command_lines` with `kmeans.labels_`, then wrapping the result with `dict(...)`.

This produces:
- Keys: each unique command string from `unique_command_lines`
- Values: the corresponding KMeans cluster label from `kmeans.labels_`

You will use this dictionary in the next cell with `.map()` to assign each original row in `dataframe` its `Cluster_KMeans` value.

Example:
```python
dict(zip(<KEYS>, <VALUES>))
```

In [ ]:
cluster_map_kmeans = dict(zip(unique_command_lines, kmeans.labels_))

Create a new column called `Cluster_KMeans` in `dataframe` by calling `.map()` on the `CommandLine` column with `cluster_map_kmeans` as the lookup.

This assigns a KMeans cluster label to **every row** in the original DataFrame — including duplicate command lines — by looking up each command in the dictionary we built in the previous cell.

Example:
```python
dataframe[<NEW MAPPED COLUMN>] = dataframe[<COLUMN TO MAP>].map(<MAPPING>)
```

In [ ]:
dataframe["Cluster_KMeans"] = dataframe["CommandLine"].map(cluster_map_kmeans)

## Step 11 - Save Clustered Data to Disk
We now have two totally different Machine Learning perspectives on our data (DBSCAN and KMeans). By mapping these final labels back to our raw logs and saving them to disk, we have successfully created a highly enriched dataset that is ready for 2D visual analysis in the next phase!
 
Save the final enriched dataset by calling `dataframe.to_parquet("_04_clustered_commands.parquet")`. 

In [ ]:
dataframe.to_parquet("_04_clustered_commands.parquet")